In [0]:
 %run /Workspace/Users/akamil92@yahoo.com/etl/Framework/etl_logger.py

In [0]:
%run /Workspace/Users/akamil92@yahoo.com/etl/Framework/utils.py

In [0]:
# ============================

from pyspark.sql.functions import col, current_timestamp, max as spark_max

table_name = "Shipments"
bronze_path = "/mnt/customer/bronze/Shipments"
source_path = "/mnt/customer/shipments.csv"

try:
    # ----------------------------
    # Load previous Bronze if exists
    # ----------------------------
    try:
        bronze_tbl = DeltaTable.forPath(spark, bronze_path)
        last_load = bronze_tbl.toDF().agg(spark_max(col("CreatedDate"))).collect()[0][0]
        first_run = False
    except:
        last_load = None
        first_run = True

    # ----------------------------
    # Read source CSV safely
    # ----------------------------
    df = read_safe_csv(source_path)

    # ----------------------------
    # Filter incremental
    # ----------------------------
    if not first_run and last_load is not None:
        df = df.filter(col("CreatedDate") > last_load)

    # ----------------------------
    # Add LastLoadTime
    # ----------------------------
    df = df.withColumn("LastLoadTime", current_timestamp())

    # ----------------------------
    # Write to Bronze Delta safely
    # ----------------------------
    write_delta_safe(df, bronze_path)

    # ----------------------------
    # Update LastLoadTime in Delta
    # ----------------------------
    try:
        update_lastload_safe(bronze_path)
    except:
        pass

    # ----------------------------
    # Log step
    # ----------------------------
    log_step(table_name, "Bronze", "Incremental Load", df_after=df)

    print(f"✓ Bronze incremental load completed → {table_name}")

except Exception as e:
    log_step(table_name, "Bronze", "Incremental Load", status="FAIL", error=str(e))
    print(f"❌ Bronze incremental load failed → {table_name}: {e}")


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

log_path = "/mnt/customer/logs/etl_log"

# Load ETL log
df_log = spark.read.format("delta").load(log_path)

# Show the most recent 50 log entries
df_log.orderBy("Timestamp", ascending=False).show(50, truncate=False)
